In [32]:
# Import ESM classes from HF transformers
from transformers import EsmTokenizer, EsmForMaskedLM, AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
import torch.nn.functional as F
import random

In [33]:
# Define path to uniref50 fasta
uniref50 = '/home/ubuntu/uniref50_subset.fasta'
# Define path to ESM model #8M parameter model

# Create a dataset class for training that comes from the uniref50 fasta
class UniRef50Dataset(Dataset):
    # Inputs are a path to the uniref50 fasta sequences, the ESM tokenizer name, and a max length of sequence cutoff
    def __init__(self, fasta_path, tokenizer_name, max_length = 1024):
        self.sequences = self.read_fasta(fasta_path)
        self.tokenizer = EsmTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length

    
    def read_fasta(self, fasta_path):
        ''' Get list of sequences from the fasta file '''
        sequences = []
        seq = []
        with open(fasta_path) as f:
            for line in f:
                line = line.strip()
                if line.startswith(">"):
                    if seq:
                        sequences.append("".join(seq))
                        seq = []
                else:
                    seq.append(line)
            if seq:
                sequences.append("".join(seq))
                
            return sequences
            
    def __len__(self):
        ''' Returns number of sequences in the dataset '''
        return len(self.sequences)

    def __getitem__(self, idx):
        ''' retrieve the tokenization of one sequence '''
        # Get a sequence
        seq = self.sequences[idx]

        # Create circular permutation of the sequence
        i = random.randint(0, len(seq)-1)
        seq_cp = seq[i:] + seq[:i]

        #Tokenize the circular permutation
        tokenized = self.tokenizer(
            seq_cp,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        #return tokenized
        return {key: val.squeeze(0) for key, val in tokenized.items()}
 

In [66]:
def generate_dataset_splits(dataset, splits = [0.70,0.15,0.15]):
    '''
    Create dataset splits
    '''
    dataset_size = len(dataset)
    train_size = int(dataset_size*splits[0])
    val_size = int(dataset_size*splits[1])
    test_size = dataset_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

    return train_dataset, val_dataset, test_dataset

def generate_dataloader(train_dataset, val_dataset, test_dataset, Batch_size):
    '''
    Generate dataloaders for dataset splits
    '''
    train_loader = DataLoader(train_dataset, batch_size=Batch_size, shuffle=True, pin_memory=True, prefetch_factor=2, num_workers=16) #pin_memory=True, prefetch_factor=2, num_workers=16 #load ahead
    val_loader = DataLoader(val_dataset, batch_size=Batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=Batch_size, shuffle=False)
    
    
    return train_loader, val_loader, test_loader

def generate_training_mask(input_ids, mask_prob = 0.15):
    '''
    Generate masked_input_ids
    input: input_ids
    output: input_ids with random 15% tokens set to masked and non-masked tokens set to -100 so they're ignored during loss calculation
    '''
    # Create set of special ids from the tokenizer which we'll not want to mask
    special_ids = {
    tokenizer.cls_token_id,
    tokenizer.pad_token_id,
    tokenizer.eos_token_id,
    tokenizer.mask_token_id,
    tokenizer.unk_token_id
}   
    # copy the input ids to a masked inputs ids tensor
    masked_input_ids = input_ids.clone()

    # Create a labels tensor used for ground truth labels during training
    ground_truth_labels = input_ids.clone()
    
    # assign random number 0 to 1 to each input id
    rand = torch.rand(input_ids.shape).to(device)
    # Create a mask; all positions less than 0.15 (mask prob) and do not mask the special tokens
    mask_arr = (rand < mask_prob) & (~input_ids.isin(torch.tensor(list(special_ids), device=input_ids.device)))
    # Apply the mask to the labels; set the masked positions equal to the tokenizer mask token id
    masked_input_ids[mask_arr] = tokenizer.mask_token_id
    # For the labels, set the tokens that are unmasked to -100 which means these tokens will not contribute to the calculation of the loss function
    ground_truth_labels[~mask_arr] = -100

    return masked_input_ids, ground_truth_labels

In [35]:
def setup_model_and_tokenizer(model_name="facebook/esm2_t6_8M_UR50D"):
    """
    Initialize tokenizer and model for fine-tuning
    """
    tokenizer = EsmTokenizer.from_pretrained(model_name)
    model = EsmForMaskedLM.from_pretrained(model_path)
    
    return model, tokenizer

In [36]:
def training_loop(model, train_loader, num_epochs=3, lr=2e-5):
    """
    Training loop for fine-tuning ESM on circularly-permuted sequences
    """
    # Move model to the GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Define optimizer; give the optimizer the model parameters and learning rate as inputs. 
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # Train for num_epochs
    for epoch in range(num_epochs):
        # Set model to train and loss to zero
        model.train()
        total_loss = 0

        # Loop through the batches in the dataloader
        for batch in train_loader:
            # Zero out the optimizer gradients
            optimizer.zero_grad()
            # Move the inputs from the batch to the GPU
            input_ids = batch['input_ids'].to(device)
            # Generate masked inputs and ground truth labels
            masked_input_ids, ground_truth_labels = generate_training_mask(input_ids, mask_prob=0.15)
            # Get the attention mask (masks out padding tokens during self-attention
            attention_mask = batch['attention_mask'].to(device)
            # Generate labels using mask function
            #labels = batch['labels'].to(device) # Loader doesn't have labels in this case

            # Get outputs from the model
            outputs = model(
                input_ids=masked_input_ids,
                attention_mask=attention_mask,
                labels=ground_truth_labels
            )

            # Calculate the loss
            loss = outputs.loss
            # Perform autograd; calculate all the gradients
            loss.backward()
            # Clip the gradients -- resizes them, makes training smoother
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # Have the optimizer take a step
            optimizer.step()
            # Add loss to total loss
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")
        
        # Validation
        model.eval()
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                masked_input_ids, ground_truth_labels = generate_training_mask(input_ids, mask_prob=0.15)
                attention_mask = batch['attention_mask'].to(device)                
                outputs = model(
                    input_ids=masked_input_ids,
                    attention_mask=attention_mask,
                    labels= ground_truth_labels
                )
                

In [37]:
dataset = UniRef50Dataset(
    fasta_path="/home/ubuntu/uniref50_subset.fasta",
    tokenizer_name="facebook/esm2_t6_8M_UR50D",
    max_length=1024
)

In [38]:
train_data, val_data, test_data = generate_dataset_splits(dataset)

In [54]:
input_ids = train_data[0]['input_ids']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = EsmTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')

rand = torch.rand(input_ids.shape).to(device)
tokenizer.mask_token_id
dataset.tokenizer.get_vocab()

In [47]:
train_data[2]

{'input_ids': tensor([ 0,  7,  9,  ..., 10,  7,  2]),
 'attention_mask': tensor([1, 1, 1,  ..., 1, 1, 1])}

In [ ]:
type(dataset

In [22]:
train_data[0]['attention_mask']

tensor([1, 1, 1,  ..., 1, 1, 1])

In [8]:
train_loader, val_loader, test_loader = generate_dataloader(train_data, val_data, test_data, Batch_size = 64)

In [16]:
train_loader.batch_size


64

In [ ]:
model, tokenizer = setup_model_and_tokenizer()

In [18]:
from prettytable import PrettyTable
def display_data_dimension_table(train_loader, val_loader, test_loader):
    train_size = len(train_loader.dataset)
    val_size = len(val_loader.dataset)
    test_size = len(test_loader.dataset)

    table = PrettyTable()
    table.field_names = ["Loader", "Dataset Size", "Batch Size", "Batches per Epoch"]
    
    table.add_row(["Train Loader", train_size, train_loader.batch_size, len(train_loader)])
    table.add_row(["Validation Loader", val_size, val_loader.batch_size, len(val_loader)])
    table.add_row(["Test Loader", test_size, test_loader.batch_size, len(test_loader)])

    print(table)

In [19]:
display_data_dimension_table(train_loader, val_loader, test_loader)

+-------------------+--------------+------------+-------------------+
|       Loader      | Dataset Size | Batch Size | Batches per Epoch |
+-------------------+--------------+------------+-------------------+
|    Train Loader   |     151      |     64     |         3         |
| Validation Loader |      32      |     64     |         1         |
|    Test Loader    |      34      |     64     |         1         |
+-------------------+--------------+------------+-------------------+


In [ ]:






#### Train the model 


optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=len(train_loader) * num_epochs
    )
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")
        
        # Validation
        model.eval()
        val_predictions, val_labels = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
                
                predictions = torch.argmax(outputs.logits, dim=-1)
                val_predictions.extend(predictions.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        val_accuracy = accuracy_score(val_labels, val_predictions)
        print(f"Validation Accuracy: {val_accuracy:.4f}")


# Set the model to train mode 
model.train()
# Move the model onto the gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
for batch in dataloader:
    input_ids = batch['input_ids']  # Shape: [batch_size, max_length]
    print(input_ids)
    print(len(input_ids))
    attention_mask = batch['attention_mask'] 

In [ ]:
### This code allows you to see the vocabulary used by the ESM tokenizer

# From the dataset object, call the get_vocab method on the tokenizer, an attribute of the dataset object and which is an object itself of class ESMtokenzier
#dataset.tokenizer.get_vocab()

In [ ]:
### This code returns total number of parameters of the model

#for param in model.parameters():
    #print(f"Shape: {param.shape}, Requires Grad: {param.requires_grad}")

#total_params = sum(p.numel() for p in model.parameters())
#print(f'The model is {model_path} and it has {total_params} parameters')

In [ ]:
### Example showing how to use the model in eval mode:
model_path = 'facebook/esm2_t6_8M_UR50D'
tokenizer_path = 'facebook/esm2_t6_8M_UR50D'

# Load an ESM model
model = EsmForMaskedLM.from_pretrained(model_path)
# Load a tokenizer


# Define an example sequence
seq = "MSKL"
# Tokenize the sequence
seq_tokenized = tokenizer(
    seq,
    return_tensors="pt",       # return PyTorch tensors
    padding="max_length",      # optional: pad to max length
    truncation=True
)
# Set the model to eval mode 
model.eval()
# Move the model onto the gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# Move the tensor of tokenized inputs onto the same device as the model
seq_tokenized = {k: v.to(device) for k, v in seq_tokenized.items()}

# Disable gradients and pass the tokenized sequence through the model
with torch.no_grad():          # disables gradients
    outputs = model(**seq_tokenized) # ** is an operator that unpacks the dictionary input_ids=seq_tokenized['input_ids'], attention_mask=seq_tokenized['attention_mask']

# Look at the outputs from the model:
logits = outputs.logits

# Apply softmax over logits to get probability distribution
probabilities = F.softmax(logits, dim=-1)


In [ ]:
    # Load tokenizer
    logger.info(f"Loading tokenizer: {args.model_name}")
    
    
    # Load model
    logger.info(f"Loading model: {args.model_name}")
    if args.task_type == 'mlm':
        model = EsmForMaskedLM.from_pretrained(args.model_name)
    else:
        model = EsmForSequenceClassification.from_pretrained(
            args.model_name, num_labels=args.num_classes
        )
    
    model.to(device)